# Zielwort-Extraktion

Lädt das zuvor in [01_model_training.ipynb](./01_model_training.ipynb) trainierte Modell, ermittelt
relevante Topics und extrahiert Zielwörter.

## Daten laden

In [ ]:
import pandas as pd
from bertopic import BERTopic

from keyword_selection.data import DATA_PATH

OUTPUT_PATH = DATA_PATH / "output"

topic_model = BERTopic.load(OUTPUT_PATH / "topic_model")
turns = pd.read_parquet(OUTPUT_PATH / "preprocessed_corpus.parquet")["rede_text"]
timestamps = turns.index.to_list()

## Zeitlicher Verlauf

Modelliert den zeitlichen Verlaufs der identifizierten Topics. Häufig (aber abhängig von den
Trainingsparametern in [01_model_training.ipynb](01_model_training.ipynb)) zeigt sich ab 2022 ein
stark ausgeprägtes Topic rund um den Russisch-Ukrainischen Konflikt.


In [ ]:
# bin count = number of years
nr_bins = len(turns.resample("YE"))

topics_over_time = topic_model.topics_over_time(
    turns.to_list(),
    timestamps,
    nr_bins=nr_bins,
)

In [ ]:
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=10)

## Wortextraktion

Zunächst wird das Ziel-Topic identifiziert (entweder manuell über die Visualisierung oder per
Query unten) und kontrolliert, ob es sich um das richtige Topic handelt.

In [ ]:
similar_topics, similarity_scores = topic_model.find_topics("Ukraine Russland Krieg", top_n=1)

seed_topic = similar_topics[0]
print(seed_topic)

topic_model.get_topic(seed_topic)[:10]

## Verwandte Topics

Anschließend werden weitere Topics ermittelt, die mit dem o.g. Topic verwandt sind. Dazu werden
alle Topics nach der semantischen (Kosinus-)Ähnlichkeit ihrer Embeddings zum Ziel-Topic gerankt und
eine gewünschte Anzahl (z.B. `TOP_N_TOPICS = 20`) selektiert.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

TOP_N_TOPICS = 20

topic_ids = sorted(topic_model.get_topics().keys())  # includes -1
embeddings = topic_model.topic_embeddings_
idx = topic_ids.index(seed_topic)

# compute cosine similarity
sims = cosine_similarity([embeddings[idx]], embeddings)[0]
similar = sorted(zip(topic_ids, sims), key=lambda x: -x[1])

# extract related topics
topic_df = topic_model.get_topic_info()
topics = topic_df[
    topic_df.Topic.isin([key for key, _ in similar[1 : 1 + TOP_N_TOPICS]])
].Topic.to_list() + [seed_topic]

Aus jedem Topic wird eine gewünschte Anzahl an Begriffen (z.B. `TOP_N_WORDS = 50`) in die finale,
nach Häufigkeit sortierte Liste von Begriffen geschrieben.

> ⚠️ Die maximal verfügbare Anzahl von Begriffen pro Topic wird **im Training** beim Aufruf von 
> `BERTopic` über den Parameter `top_n_words` gesteuert. Bei Bedarf kann das Training in
> [01_model_training.ipynb](./01_model_training.ipynb) mit höheren Werten wiederholt werden.

In [ ]:
from keyword_selection.data import DATA_PATH

TOP_N_WORDS = 50

(
    pd.Series(topics)
    .map(topic_model.get_topic)
    .str[:TOP_N_WORDS]
    .explode()
    .str[0]
    .value_counts()
    .to_csv(DATA_PATH / "output" / "keywords.csv", header=None)
)
